
## Gaussian Elimination and LU Factorization (Solving $Ax=b$)

When you need to solve a linear system

\[
A x = b,
\]

a common beginner move is to compute $A^{-1}$ and then set $x = A^{-1}b$.
In numerical computing, **this is usually a bad idea**:

- Computing an explicit inverse costs extra work and can amplify numerical error.
- The preferred approach is to **solve** the system directly using **Gaussian elimination**.

### Gaussian elimination (idea)
Gaussian elimination uses row operations to transform the system into an equivalent **upper-triangular** system:

\[
U x = c,
\]

which can then be solved efficiently by **back substitution**.

### LU factorization (key concept)
The elimination steps can be “saved” in a factorization:

\[
A = L U,
\]

where:

- $L$ is **lower triangular** with ones on the diagonal (Doolittle form),
- $U$ is **upper triangular**.

With pivoting (recommended), we use a permutation matrix $P$:

\[
P A = L U.
\]

Then solving $Ax=b$ becomes:

1. Solve \(L y = P b\) by **forward substitution**  
2. Solve \(U x = y\) by **back substitution**

This is the workhorse behind many `solve()` routines.

**Question.** Why is solving \(Ax=b\) via `np.linalg.solve(A,b)` generally preferred over forming `np.linalg.inv(A) @ b`?


In [ ]:
import numpy as np



### A small example (NumPy)

We’ll solve a simple system two ways:

- the recommended way: `np.linalg.solve(A,b)`
- the “inverse way” (for comparison only)

Observe that both return (nearly) the same answer, but `solve` is the method you should use in practice.


In [ ]:
A = np.array([[2., 1., 1.],
              [4., -6., 0.],
              [-2., 7., 2.]])

b = np.array([5., -2., 9.])

x_solve = np.linalg.solve(A, b)
x_inv   = np.linalg.inv(A) @ b

print("x from solve:", x_solve)
print("x from inv @ b:", x_inv)

print("\nResidual ||Ax-b||:", np.linalg.norm(A @ x_solve - b))



## LU factorization with partial pivoting

Below you will implement LU factorization (Doolittle) **with partial pivoting**.

- **Partial pivoting** swaps rows so the pivot element is as large as possible in magnitude.
- This greatly improves numerical stability.
- The factorization is \(P A = L U\).

We will represent the permutation matrix \(P\) as a 1D integer array `p` such that  
`(P @ A) == A[p, :]` and `(P @ b) == b[p]`.

### What you will write

1. `lu_factor(A)` returning `(p, L, U)`  
2. `forward_sub(L, b)` solving \(Ly=b\)  
3. `back_sub(U, y)` solving \(Ux=y\)  
4. `lu_solve(p, L, U, b)` solving \(Ax=b\)

**Important.** Do not call `np.linalg.solve` or `np.linalg.inv` inside your implementations.


In [ ]:
def lu_factor(A, tol=1e-12):
    """
    Compute an LU factorization with partial pivoting:
        P A = L U
    using the Doolittle convention (diag(L) = 1).

    Parameters
    ----------
    A : np.ndarray (n, n)
    tol : float
        pivot tolerance for singular detection

    Returns
    -------
    p : np.ndarray (n,)
        permutation vector (row order)
    L : np.ndarray (n, n)
        unit lower triangular
    U : np.ndarray (n, n)
        upper triangular

    Raises
    ------
    ValueError if A is not square or (numerically) singular.
    """
    import numpy as np

    A = np.array(A, dtype=float)
    n, m = A.shape
    if n != m:
        raise ValueError("A must be square")

    U = A.copy()
    L = np.eye(n)
    p = np.arange(n)

    for k in range(n):
        # TODO 1: choose pivot row r >= k that maximizes abs(U[r, k])
        # r = ...
        r = None  # replace

        # TODO 2: swap rows in U (and record in p). Also swap the relevant part of L.
        # - swap U[[k, r], :] 
        # - swap p[[k, r]]
        # - swap L[[k, r], :k]  (ONLY columns < k, because later columns not set yet)

        # TODO 3: singular check
        # if abs(U[k, k]) < tol: raise ValueError("Singular matrix")

        # TODO 4: elimination below pivot
        # for i in range(k+1, n):
        #     L[i, k] = U[i, k] / U[k, k]
        #     U[i, k:] -= L[i, k] * U[k, k:]

        pass  # remove after completing the loop

    return p, L, U


def forward_sub(L, b):
    """Solve Ly=b for y, where L is lower triangular with diag(L)=1."""
    import numpy as np
    L = np.array(L, dtype=float)
    b = np.array(b, dtype=float)
    n = L.shape[0]
    y = np.zeros(n)

    for i in range(n):
        # TODO: y[i] = b[i] - sum_{j < i} L[i,j] y[j]
        pass

    return y


def back_sub(U, y, tol=1e-12):
    """Solve Ux=y for x, where U is upper triangular."""
    import numpy as np
    U = np.array(U, dtype=float)
    y = np.array(y, dtype=float)
    n = U.shape[0]
    x = np.zeros(n)

    for i in range(n-1, -1, -1):
        # TODO 1: singular check on U[i,i]
        # TODO 2: x[i] = (y[i] - sum_{j > i} U[i,j] x[j]) / U[i,i]
        pass

    return x


def lu_solve(p, L, U, b):
    """Solve Ax=b given p,L,U from lu_factor(A)."""
    import numpy as np
    b = np.array(b, dtype=float)
    bp = b[p]            # Apply permutation: Pb = b[p]
    y = forward_sub(L, bp)
    x = back_sub(U, y)
    return x



### Sanity check (after you finish the TODOs)

Run the cell below after implementing the functions.
You should see a small residual and close agreement with `np.linalg.solve`.


In [ ]:
# Uncomment and run after you finish the TODOs

# import numpy as np
# A = np.array([[2., 1., 1.],
#               [4., -6., 0.],
#               [-2., 7., 2.]])
# b = np.array([5., -2., 9.])

# p, L, U = lu_factor(A)
# x = lu_solve(p, L, U, b)

# print("x (LU)   :", x)
# print("x (solve):", np.linalg.solve(A, b))
# print("||Ax-b|| :", np.linalg.norm(A @ x - b))



## Exercise (coding + interpretation)

### (1) Coding test
After your implementation is complete, run the following tests:

1. Generate a random matrix \(A\in\mathbb{R}^{n\times n}\) (e.g. \(n=8\)) and a random vector \(b\).
2. Use your LU solver to compute \(x_{\text{LU}}\).
3. Compare to NumPy’s solution \(x_{\text{np}}\) via `np.linalg.solve`.
4. Report:

- \(\|x_{\text{LU}} - x_{\text{np}}\|_\infty\)
- \(\|Ax_{\text{LU}} - b\|_2\)

### (2) Pivoting experiment
Create a matrix that is *almost* singular or has a tiny pivot (hint: start with an identity matrix and make two rows nearly dependent).
Compare what happens **with** and **without** partial pivoting (you can temporarily disable pivoting by forcing `r=k`).

**Deliverable.** Submit your code and a short paragraph explaining what you observed and why pivoting matters.


In [ ]:
# Exercise starter: numerical tests

# import numpy as np
# rng = np.random.default_rng(0)
# n = 8
# A = rng.normal(size=(n, n))
# b = rng.normal(size=n)

# p, L, U = lu_factor(A)
# x_lu = lu_solve(p, L, U, b)
# x_np = np.linalg.solve(A, b)

# print("||x_lu - x_np||_inf =", np.linalg.norm(x_lu - x_np, ord=np.inf))
# print("||A x_lu - b||_2    =", np.linalg.norm(A @ x_lu - b))

# # Pivoting experiment hint matrix:
# eps = 1e-10
# A_bad = np.eye(5)
# A_bad[1, :] = A_bad[0, :] + eps  # nearly dependent row
# b_bad = rng.normal(size=5)

# # Try your solver on A_bad:
# # p, L, U = lu_factor(A_bad)
# # x_bad = lu_solve(p, L, U, b_bad)
# # print("||A_bad x - b_bad||_2 =", np.linalg.norm(A_bad @ x_bad - b_bad))
